# Statistical arbitrage with Cointegration

## Pairs Trading & Statistical Arbitrage

Statistical arbitrage refers to strategies that employ some statistical model or method to take
advantage of what appears to be relative mispricing of assets, while maintaining a level of
market neutrality.

Pairs trading is a conceptually straightforward strategy that has been employed by algorithmic traders since at least the mid-eighties ([Gatev, Goetzmann, and Rouwenhorst 2006](http://www-stat.wharton.upenn.edu/~steele/Courses/434/434Context/PairsTrading/PairsTradingGGR.pdf)). The goal is to find two assets whose prices have historically moved together, track the spread (the difference between their prices), and, once the spread widens, buy the
loser that has dropped below the common trend and short the winner. If the relationship persists, the long and/or the short leg will deliver profits as prices converge and the positions are closed.

This approach extends to a multivariate context by forming baskets from multiple securities and trading one asset against a basket of two baskets against each other.

## Pairs Trading in Practice

In practice, the strategy requires two steps:

1. **Formation phase**: Identify securities that have a long-term mean-reverting relationship. Ideally, the spread should have a high variance to allow for frequent profitable trades while reliably reverting to the common trend.
2. **Trading phase**: Trigger entry and exit trading rules as price movements cause thespread to diverge and converge.

Several approaches to the formation and trading phases have emerged from increasingly active research in this area, across multiple asset classes, over the last several years. The book outlines the key differences between them; the notebook dives into an example application.

## Imports & Settings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from collections import Counter

from time import time
from pathlib import Path

import numpy as np
import pandas as pd

from pykalman import KalmanFilter
from statsmodels.tsa.stattools import coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.api import VAR

import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

In [ ]:
idx = pd.IndexSlice
sns.set_style('whitegrid')

In [ ]:
def format_time(t):
    m_, s = divmod(t, 60)
    h, m = divmod(m_, 60)
    return f'{h:>02.0f}:{m:>02.0f}:{s:>02.0f}'

### Johansen Test Critical Values

In [ ]:
critical_values = {0: {.9: 13.4294, .95: 15.4943, .99: 19.9349},
                   1: {.9: 2.7055, .95: 3.8415, .99: 6.6349}}

In [ ]:
trace0_cv = critical_values[0][.95] # critical value for 0 cointegration relationships
trace1_cv = critical_values[1][.95] # critical value for 1 cointegration relationship

## Load Data

In [ ]:
symbol_list = [
    'AGQ', 'BBY', 'CSX', 'DAL', 'EPP', 'EPU', 'EWA', 'EWD', 'EWJ', 'HEDJ',
    'HPQ', 'LMT', 'SPY', 'STT', 'SWKS', 'TECL', 'THD', 'TIP', 'WBA', 'XOM'
]

start_date = '2015-01-01'
end_date = '2017-06-30'

data = yf.download(
    symbol_list, 
    start=start_date, 
    end=end_date,
    auto_adjust=False
)['Adj Close']

etfs_symbols = ['AGQ', 'EPP', 'EPU', 'EWA', 'EWD', 'EWJ', 'HEDJ', 'SPY', 'TECL', 'THD', 'TIP']
stocks_symbols = ['BBY', 'CSX', 'DAL', 'HPQ', 'LMT', 'STT', 'SWKS', 'WBA', 'XOM']

etfs = data[etfs_symbols].loc['2015':]
etfs.info()

stocks = data[stocks_symbols].loc['2015':]
stocks.info()

In [ ]:
tickers = {}
for symbol in symbol_list:
    info = yf.Ticker(symbol).info
    tickers[symbol] = info.get('longName', info.get('shortName', symbol))

names = tickers

pd.Series(names).count()

## Precompute Cointegration

In [ ]:
def test_cointegration(etfs, stocks, test_end, lookback=2):
    start = time()
    results = []
    test_start = test_end - pd.DateOffset(years=lookback) + pd.DateOffset(days=1)
    
    print(f"Test Start Date: {test_start.strftime('%Y-%m-%d')}")
    print(f"Test End Date  : {test_end.strftime('%Y-%m-%d')}")
    
    etf_tickers = etfs.columns.tolist()
    etf_data = etfs.loc[str(test_start):str(test_end)]

    stock_tickers = stocks.columns.tolist()
    stock_data = stocks.loc[str(test_start):str(test_end)]
    
    print(f"Number of samples (days) in range: {len(etf_data)}")
    
    n = len(etf_tickers) * len(stock_tickers)
    j = 0
    for i, s1 in enumerate(etf_tickers, 1):
        for s2 in stock_tickers:
            j += 1
            if j % 1000 == 0:
                print(f'\t{j:5,.0f} ({j/n:3.1%}) | {time() - start:.2f}')
            df = etf_data.loc[:, [s1]].dropna().join(stock_data.loc[:, [s2]].dropna(), how='inner')
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                var = VAR(df)
                lags = var.select_order()
                result = [test_end, s1, s2]
                order = lags.selected_orders['aic']
                result += [coint(df[s1], df[s2], trend='c')[1], coint(df[s2], df[s1], trend='c')[1]]

            cj = coint_johansen(df, det_order=0, k_ar_diff=order)
            result += (list(cj.lr1) + list(cj.lr2) + list(cj.evec[:, cj.ind[0]]))
            results.append(result)
    return results

### Define Test Periods

In [ ]:
dates = stocks.loc['2016-12':'2019-6'].resample('Q').last().index
dates

### Run Tests

In [ ]:
# PARALLEL VERSION
from joblib import Parallel, delayed

test_results = []
columns = ['test_end', 's1', 's2', 'eg1', 'eg2',
           'trace0', 'trace1', 'eig0', 'eig1', 'w1', 'w2']

# Parallelize the loop over dates
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(test_cointegration)(etfs, stocks, test_end=test_end) for test_end in dates
)

# Convert each result to a DataFrame
test_dfs = [pd.DataFrame(result, columns=columns) for result in results]

# Concatenate and save
test_results = pd.concat(test_dfs)

#### Reload  Test Results

Column Definitions for Cointegration Tests
==========================================

The list `columns = ['test_end', 's1', 's2', 'eg1', 'eg2', 'trace0', 'trace1', 'eig0', 'eig1', 'w1', 'w2']` defines columns for a DataFrame storing results from cointegration tests on ETF-stock pairs, run in `test_cointegration` over a 2-year lookback period ending at `test_end`. Cointegration indicates a stable long-term relationship between non-stationary time series, useful for pairs trading. Tests include Engle-Granger (EG) and Johansen methods.

-   **test_end**: End date of the test period (quarterly, 2016-12 to 2019-06). Input to `test_cointegration`.
-   **s1**: ETF ticker (e.g., 'SPY'). From `etfs` DataFrame columns.
-   **s2**: Stock ticker (e.g., 'AAPL'). From `stocks` DataFrame columns.
-   **eg1**: P-value from EG test with s1 as dependent (regressed on s2). From `coint(df[s1], df[s2], trend='c')[1]`. Low p-value (<0.05) suggests cointegration.
-   **eg2**: P-value from EG test with s2 as dependent (regressed on s1). From `coint(df[s2], df[s1], trend='c')[1]`. Checks reverse direction.
-   **trace0**: Johansen trace statistic for H0: r=0 (no cointegration). From `coint_johansen(df, det_order=0, k_ar_diff=order).lr1[0]`. If > 15.4943 (95% critical value), reject H0, indicating cointegration.
-   **trace1**: Johansen trace statistic for H0: r≤1. From `coint_johansen(...).lr1[1]`. For pairs, r>1 is impossible (saturation), so typically < 3.8415 (95% critical value), confirming at most one relation.
-   **eig0**: Johansen max-eigenvalue statistic for r=0. From `coint_johansen(...).lr2[0]`. High values suggest cointegration.
-   **eig1**: Johansen max-eigenvalue statistic for r=1. From `coint_johansen(...).lr2[1]`. Tests impossible r=2, usually insignificant.
-   **w1**: Cointegrating vector weight for s1. From `coint_johansen(...).evec[:, cj.ind[0]][0]`. Part of β vector for stationary combination.
-   **w2**: Cointegrating vector weight for s2. From `coint_johansen(...).evec[:, cj.ind[0]][1]`. With `w1`, forms hedge ratio for trading.

In [ ]:
test_results.info()
test_results

## Identify Cointegrated Pairs

### Significant Johansen Trace Statistic

#### Notes on `trace0` and `trace1`

For pairs (two series), the Johansen test has a maximum rank of 1. `trace0` tests for cointegration (r>0); `trace1` tests for r>1 (impossible, hence "saturation"). A high `trace0` with low `trace1` suggests exactly one cointegrating relation.

In [ ]:
test_results['joh_sig'] = ((test_results.trace0 > trace0_cv) &
                           (test_results.trace1 < trace1_cv))

In [ ]:
# test_results['joh_sig'] = ((test_results.trace0 > trace0_cv) &
#                            (test_results.trace1 > trace1_cv))

In [ ]:
test_results.joh_sig.value_counts(normalize=True)

### Significant Engle Granger Test

In [ ]:
test_results['eg'] = test_results[['eg1', 'eg2']].min(axis=1)
test_results['s1_dep'] = test_results.eg1 < test_results.eg2
test_results['eg_sig'] = (test_results.eg < .05)

In [ ]:
test_results.eg_sig.value_counts(normalize=True)

### Comparison Engle-Granger vs Johansen

In [ ]:
test_results['coint'] = (test_results.eg_sig & test_results.joh_sig)
test_results.coint.value_counts(normalize=True)

In [ ]:
test_results = test_results.drop(['eg1', 'eg2', 'trace0', 'trace1', 'eig0', 'eig1'], axis=1)
test_results.info()

In [ ]:
test_results

### Reading the Output
- **Proportion Column**: This is the fraction (mean of boolean `coint` values) of tested ETF-stock pairs that are cointegrated per quarterly `test_end` date (e.g., 0.055761 means ~5.58% of pairs passed both Engle-Granger (p<0.05) and Johansen (trace stats > critical values) tests for data ending 2016-12-31).
- **Overall Trend**: Proportions range from ~1.92% (2019-06-30) to ~6.86% (2017-06-30), averaging ~3.7% across periods, showing variability in cointegrated pairs over time.

In [ ]:
def select_candidate_pairs(data):
    candidates = data[data.joh_sig | data.eg_sig]
    candidates['y'] = candidates.apply(lambda x: x.s1 if x.s1_dep else x.s2, axis=1)
    candidates['x'] = candidates.apply(lambda x: x.s2 if x.s1_dep else x.s1, axis=1)
    return candidates.drop(['s1_dep', 's1', 's2'], axis=1)

candidates = select_candidate_pairs(test_results)

In [ ]:
proportion_data = test_results.groupby('test_end').coint.mean().to_frame('Proportion')
print(proportion_data)

In [ ]:
ax = test_results.groupby('test_end').coint.mean().to_frame('Proportion').plot()
ax.axhline(.05, lw=1, ls='--', c='k');

In [ ]:
proportion_data = test_results.groupby('test_end').coint.mean().to_frame('Proportion')
proportion_data['Num_Pairs'] = candidates.groupby('test_end').size()
proportion_data['Cointegrated_Pairs'] = (proportion_data['Proportion'] * proportion_data['Num_Pairs']).round()
# Create interactive bar plot
import plotly.express as px
fig = px.bar(
    proportion_data,
    x=proportion_data.index,
    y='Proportion',
    title='Proportion of Cointegrated Pairs by Test Period',
    labels={'test_end': 'Test End Date', 'Proportion': 'Proportion Cointegrated'},
    text='Proportion'  # Display proportion on bars
)
fig.update_traces(texttemplate='%{text:.2%}', textposition='auto')  # Format as percentage
fig.add_hline(y=0.05, line_dash='dash', line_color='black', annotation_text='5% Threshold')
fig.update_layout(
    yaxis_tickformat='.0%',  # Show y-axis as percentage
    showlegend=False,
    plot_bgcolor='white',
    bargap=0.2
)
fig.show()

# Create styled table
styled_table = proportion_data.style.format({
    'Proportion': '{:.2%}',
    'Num_Pairs': '{:.0f}',
    'Cointegrated_Pairs': '{:.0f}'
}).background_gradient(subset=['Proportion'], cmap='RdYlGn', low=0, high=0.5)
display(styled_table)

### Select Candidate Pairs

In [ ]:
candidates.info()
candidates

#### Candidates over Time

In [ ]:
candidates.groupby('test_end').size().plot(figsize=(8, 5))

#### Most Common Pairs 

In [ ]:
prices = data.ffill(limit=5)

In [ ]:
# Get the top 10 most common pairs
most_common_pairs = pd.DataFrame(counter.most_common(10))
most_common_pairs = pd.DataFrame(most_common_pairs[0].values.tolist(), columns=['s1', 's2'])
most_common_pairs

In [ ]:
from collections import Counter

# Create Counter to tally frequency of cointegrated pairs
counter = Counter()
for s1, s2 in zip(candidates[candidates.joh_sig & candidates.eg_sig].y, 
                  candidates[candidates.joh_sig & candidates.eg_sig].x):
    if s1 > s2:
        counter[(s2, s1)] += 1
    else: 
        counter[(s1, s2)] += 1
        
counter

In [ ]:
cnt = pd.Series(counter).reset_index()
cnt.columns = ['s1', 's2', 'n']
cnt['name1'] = cnt.s1.map(tickers)
cnt['name2'] = cnt.s2.map(tickers)
cnt.nlargest(10, columns='n')

The `counter = Counter()` creates a `Counter` object that tallies the frequency of each unique sorted pair `(ticker1, ticker2)` (with `ticker1 < ticker2` lexicographically) among the candidate pairs where both `joh_sig` (Johansen significance) and `eg_sig` (Engle-Granger significance) are True.

Each count represents the number of test periods (out of the 11 quarterly periods in the provided `DatetimeIndex`) in which that pair demonstrated cointegration according to both tests.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for i in range(len(most_common_pairs)):
    # Get the tickers for the current pair
    s1, s2 = most_common_pairs.at[i, 's1'], most_common_pairs.at[i, 's2']
    
    # Create a new figure for each pair
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # Plot the price series for s1 and s2
    prices.loc[:, [s1, s2]].rename(columns=tickers).plot(
        secondary_y=tickers[s2],  # Second ticker on right y-axis
        ax=ax,
        rot=0  # Horizontal x-axis labels
    )
    
    # Customize the plot
    ax.grid(False)
    ax.set_xlabel('')  # Remove x-axis label
    ax.set_title(f'Price Series: {tickers[s1]} vs {tickers[s2]}')  # Add title with pair names
    
    # Clean up with Seaborn style
    sns.despine()
    plt.tight_layout()
    
    # Show the plot
    plt.show()

## Get Entry and Exit Dates 

### Explanation of Key Functions in the Pairs Trading Code

This code is part of a strategy for "pairs trading," where you find two related assets (like an ETF and a stock) whose prices usually move together. When they temporarily drift apart (diverge), you bet they'll come back together (converge) by buying one and selling the other. The functions below help prepare the data for this: smoothing noisy prices, figuring out how to balance trades, estimating how quickly divergences fix themselves, and processing all pairs efficiently. I'll explain each in plain English, with examples tied to the code's context (e.g., using ETF 'SPY.US' and stock 'XOM.US' as a sample pair).

#### KFSmoother: Smoothing Prices to Reduce Noise
This function uses a mathematical tool called a Kalman filter to "smooth" out the daily ups and downs in asset prices, making it easier to see the true underlying trend. It's like applying a smart moving average that adapts over time, ignoring short-term wiggles caused by market noise.

- **Plain English Breakdown**:
  - Prices in financial markets are bumpy due to random events (e.g., news or trades). This function estimates a cleaner version of the price series by assuming the "true" price evolves gradually but gets observed with some error.
  - It starts with an initial guess (mean=0) and updates its estimate step by step as new price data comes in, balancing between the observed price and its prediction.
  - The result is a smoothed series that's less volatile, which helps in later steps like calculating relationships between two assets.

- **How It Works in Code**:
  - It sets up a simple Kalman filter model: The price doesn't change much from day to day (transition matrix is identity), but there's a little wiggle room (covariance=0.05 for changes, 1 for observation noise).
  - It runs the filter on the price values and returns a new series with the smoothed estimates.

- **Example in Context**:
  - Imagine raw prices for 'SPY.US' (S&P 500 ETF) over a week: [200, 202, 198, 205, 199]. These jump around.
  - After smoothing: [200, 201, 200, 202, 201]. It's steadier, reducing noise.
  - In the code, `smoothed_prices = prices.apply(KFSmoother)` applies this to every ticker's column in the `prices` DataFrame (e.g., all ETFs and stocks from 2016-2019). This preprocessed data is then used for hedge ratios and spreads, making the strategy more reliable than using raw, noisy prices.

In [ ]:
def KFSmoother(prices):
    """Estimate rolling mean"""
    
    kf = KalmanFilter(transition_matrices=np.eye(1),
                      observation_matrices=np.eye(1),
                      initial_state_mean=0,
                      initial_state_covariance=1,
                      observation_covariance=1,
                      transition_covariance=.05)

    state_means, _ = kf.filter(prices.values)
    return pd.Series(state_means.flatten(),
                     index=prices.index)

#### KFHedgeRatio: Calculating a Dynamic Balance for Trading Pairs
This function figures out the "hedge ratio" – basically, how much of one asset you need to trade against the other to keep the pair balanced and neutral to overall market moves. It's dynamic, meaning it changes over time as market conditions shift, using another Kalman filter for adaptability.

- **Plain English Breakdown**:
  - In pairs trading, you don't just buy/sell equal amounts; one asset might be twice as volatile as the other, so you need a ratio (e.g., sell 2 units of Asset X for every 1 unit of Asset Y you buy) to cancel out common movements.
  - This function treats the relationship as a changing linear equation (Y ≈ β * X + intercept), estimating β (the ratio) and the intercept over time.
  - It returns a negative version of the estimates because the code later uses it to build the spread as Y + (negative β) * X, which is equivalent to Y - β * X.

- **How It Works in Code**:
  - It sets up a Kalman filter for a 2D state (β and intercept): They evolve slowly (small transition covariance based on delta=0.001).
  - The observation matrix uses X and a constant (for intercept), and it filters based on Y's values.
  - Output: A time series of -[β, intercept] for each day.

- **Example in Context**:
  - For pair Y='SPY.US' (dependent, price ~200) and X='XOM.US' (independent, price ~80):
    - On Day 1, it might estimate β ≈ 2.5 (meaning SPY moves 2.5 times more than XOM in response to common factors).
    - Hedge ratio returned: -2.5 (negative for spread formula).
    - In the code, inside `process_pair`: `KFHedgeRatio(y=smoothed SPY prices, x=smoothed XOM prices)[:, 0]` gives the daily -β, used to compute the spread. If β=2.5, you might short 2.5 shares of XOM per share of SPY to hedge.

In [ ]:
def KFHedgeRatio(x, y):
    """Estimate Hedge Ratio and Intercept"""
    delta = 1e-3
    trans_cov = delta / (1 - delta) * np.eye(2)
    obs_mat = np.expand_dims(np.vstack([[x], [np.ones(len(x))]]).T, axis=1)

    kf = KalmanFilter(n_dim_obs=1, n_dim_state=2,
                      initial_state_mean=[0, 0],
                      initial_state_covariance=np.ones((2, 2)),
                      transition_matrices=np.eye(2),
                      observation_matrices=obs_mat,
                      observation_covariance=2,
                      transition_covariance=trans_cov)

    state_means, _ = kf.filter(y.values)
    hedge_ratios = -state_means[:, 0]
    intercepts = -state_means[:, 1]
    return hedge_ratios, intercepts

### Estimate mean reversion half life


#### estimate_half_life: Estimating How Long Divergences Take to Fix Themselves
This function calculates the "half-life" of a price spread – the average number of days it takes for a divergence between two assets to shrink by half, assuming it mean-reverts (comes back to normal). It's a measure of how quickly the pair corrects itself, which helps decide trading windows.

- **Plain English Breakdown**:
  - Mean-reverting spreads don't snap back instantly; they take time. Half-life tells you the speed: Short (e.g., 10 days) means fast fixes (good for quick trades); long (e.g., 200 days) means slow (riskier, as things might change).
  - It fits a simple regression model to the spread's changes, estimating the reversion speed (beta), then converts it to days using a formula ($-ln(2)/beta$).
  - Ensures a minimum of 1 day to avoid nonsense values.

- **How It Works in Code**:
  - Creates lagged spread (X) and differences (Y), adds a constant for intercept.
  - Solves for beta using linear algebra (normal equation for OLS regression).
  - Computes half-life and rounds/clamps it.

- **Example in Context**:
  - Suppose the spread (SPY - β * XOM) over 2 years: Starts at 0, jumps to 4, then slowly returns (e.g., 4 → 2 in 20 days, 2 → 1 in another 20).
  - Beta might be -0.035 (negative indicates reversion), half-life ≈ 20 days (-0.693 / -0.035 ≈ 20).
  - In the code, inside `process_pair`: `half_life = estimate_half_life(pair.spread.loc[t: test_end])` uses the formation period (2 years pre-trading). This 20-day value sets the rolling window for z-scores (min(40, max_window)), helping detect tradable divergences.

In [ ]:
def estimate_half_life(spread):
    X = spread.shift().iloc[1:].to_frame().assign(const=1)
    y = spread.diff().iloc[1:]
    beta = (np.linalg.inv(X.T @ X) @ X.T @ y).iloc[0]
    halflife = int(round(-np.log(2) / beta, 0))
    return max(halflife, 1)

### Compute Spread & Bollinger Bands

#### get_spread_parallel: Processing All Pairs Efficiently in Parallel
This is the main workhorse function that loops over time periods and candidate pairs, computing everything needed for trading (hedge ratios, spreads, half-lives, z-scores) using the above helpers. It runs in parallel to speed things up, as there are thousands of pairs.

- **Plain English Breakdown**:
  - It breaks the data into quarterly "test periods" (e.g., ending Dec 2016), grabs candidate pairs for each, and for a 2-year "formation" window plus 6-month "trading" window:
    - Smooths prices, computes daily hedge ratios and spreads.
    - Estimates half-life to size a rolling window.
    - Calculates z-scores (how far the spread is from normal, in standard deviations) for spotting trades (e.g., z>2 means diverge, enter trade).
  - Parallelizes per-pair work to handle scale (e.g., 1000+ pairs/period) quickly.

- **How It Works in Code**:
  - Outer loop: Over unique test_end dates (quarters from 2016-2019).
  - For each period: Define time windows (t=2 years back, T=6 months forward).
  - Inner function `process_pair`: For each pair (y,x), compute smoothed prices, hedge ratio, spread, half-life, rolling mean/std, z-score; output trading-period DataFrame and half-life list.
  - Uses `joblib.Parallel` to run `process_pair` on all CPUs.
  - Collects results into lists: `pairs` (DataFrames per pair-period) and `half_lives` (lists per pair).

- **Example in Context**:
  - For period ending 2016-12-31, with 1000 candidates (e.g., SPY-XOM as pair 1).
    - Formation: 2015-01-01 to 2016-12-31; Trading: 2017-01-01 to 2017-06-30.
    - For SPY-XOM: Smooth prices, get hedge ratios (e.g., -2.89 on Jan 3), spread (e.g., 2.50), half-life (19 days), z-score (e.g., -0.61 – not extreme).
    - Outputs: A 125-row DataFrame (trading days) with these metrics for SPY-XOM, plus half-life [2016-12-31, 'SPY.US', 'XOM.US', 19].
  - Full run: Produces `pairs` (list of ~thousands of such DataFrames) and `half_lives` (list of lists), used later for trade signals (e.g., enter if |z|>2).

These functions work together: Smoothing cleans data, hedge ratio balances pairs, half-life tunes timing, and parallel processing makes it feasible for many pairs. In the strategy, this setup identifies profitable convergence opportunities while managing risk.

In [ ]:
def get_spread(candidates, prices):
    pairs = []
    half_lives = []

    periods = pd.DatetimeIndex(sorted(candidates.test_end.unique()))
    start = time()
    for p, test_end in enumerate(periods, 1):
        start_iteration = time()

        period_candidates = candidates.loc[candidates.test_end == test_end, ['y', 'x']]
        trading_start = test_end + pd.DateOffset(days=1)
        t = trading_start - pd.DateOffset(years=2)
        T = trading_start + pd.DateOffset(months=6) - pd.DateOffset(days=1)
        max_window = len(prices.loc[t: test_end].index)
        print(f"max window: {max_window}")
        print(f"test_end {test_end.date()}, {len(period_candidates)} pairs")
        for i, (y, x) in enumerate(zip(period_candidates.y, period_candidates.x), 1):
            if i % 1000 == 0:
                msg = f'{i:5.0f} | {time() - start_iteration:7.1f} | {time() - start:10.1f}'
                print(msg)
            pair = prices.loc[t: T, [y, x]]
            pair['hedge_ratio'] = KFHedgeRatio(y=KFSmoother(prices.loc[t: T, y]),
                                               x=KFSmoother(prices.loc[t: T, x]))[:, 0]
            pair['spread'] = pair[y].add(pair[x].mul(pair.hedge_ratio))
            half_life = estimate_half_life(pair.spread.loc[t: test_end])                
            spread = pair.spread.rolling(window=min(2 * half_life, max_window))
            print(f"half_life {half_life} , spread window {min(2 * half_life, max_window)}")
            pair['z_score'] = pair.spread.sub(spread.mean()).div(spread.std())
            pairs.append(pair.loc[trading_start: T].assign(s1=y, s2=x, period=p, pair=i).drop([x, y], axis=1))

            half_lives.append([test_end, y, x, half_life])
    return pairs, half_lives

# %%
candidates.info()

48m 20.0s


- **Max window:** 252, **Test end:** 2016-12-31, **Pairs:** 3497
- **Max window:** 314, **Test end:** 2017-03-31, **Pairs:** 1978
- **Max window:** 377, **Test end:** 2017-06-30, **Pairs:** 4124
- **Max window:** 440, **Test end:** 2017-09-30, **Pairs:** 2024
- **Max window:** 503, **Test end:** 2017-12-31, **Pairs:** 2885
- **Max window:** 503, **Test end:** 2018-03-31, **Pairs:** 3513
- **Max window:** 503, **Test end:** 2018-06-30, **Pairs:** 2399
- **Max window:** 502, **Test end:** 2018-09-30, **Pairs:** 2929
- **Max window:** 502, **Test end:** 2018-12-31, **Pairs:** 2846
- **Max window:** 501, **Test end:** 2019-03-31, **Pairs:** 2606
- **Max window:** 501, **Test end:** 2019-06-30, **Pairs:** 2645


```python
print(y, x) 
#SPY.US XOM.US
```

```python
pair = prices.loc[t: T, [y, x]]
print(pair)
ticker         BMY.US     ILF.US
date                            
2016-07-01  60.032011  20.127575
2016-07-05  60.274389  20.160980
2016-07-06  60.481711  20.160264
2016-07-07  60.683829  20.138191
2016-07-08  61.045403  20.273513
...               ...        ...
2018-12-24  44.698739  25.344163
2018-12-26  44.580871  25.357431
2018-12-27  44.553777  25.403545
2018-12-28  44.625562  25.489196
2018-12-31  44.866089  25.577877
```

In [ ]:
# pairs, half_lives = get_spread(candidates, smoothed_prices)

5m 4.0s

In the `get_spread_parallel` function, the variables `trading_start`, `t`, and `T` define key dates for analyzing cointegrated pairs in a statistical arbitrage strategy:

- **`trading_start`**: This is the date when trading begins for a given test period. It’s set to the day after the `test_end` date, marking the start of the trading window.
- **`t`**: This is the start of the lookback period, exactly two years before `trading_start`. It defines the beginning of the historical data used to estimate the spread and hedge ratio.
- **`T`**: This is the end of the trading window, six months after `trading_start`. It marks the cutoff date for the trading period being analyzed.

In plain terms, `trading_start` is when you start trading, `t` is the start of the two-year historical data window used for calculations, and `T` is the end of the six-month trading period.

In [ ]:
def get_spread_parallel(candidates, prices):
    pairs = []
    half_lives = []

    periods = pd.DatetimeIndex(sorted(candidates.test_end.unique()))
    start = time()
    for p, test_end in enumerate(periods, 1):
        start_iteration = time()

        period_candidates = candidates.loc[candidates.test_end == test_end, ['y', 'x']]
        trading_start = test_end + pd.DateOffset(days=1) #test end date + 1
        t = trading_start - pd.DateOffset(years=2) # 2 years before trading start
        T = trading_start + pd.DateOffset(months=6) - pd.DateOffset(days=1) # 6 months after trading start - 1 day
        max_window = len(prices.loc[t: test_end].index)
        print(test_end.date(), len(period_candidates))

        def process_pair(i, y, x):
            pair = prices.loc[t: T, [y, x]]
            hedge_ratio, intercept = KFHedgeRatio(y=KFSmoother(prices.loc[t: T, y]),
                                                  x=KFSmoother(prices.loc[t: T, x]))
            
            pair['hedge_ratio'] = hedge_ratio
            pair['intercept'] = intercept
            pair['spread'] = pair[y].add(pair[x].mul(pair['hedge_ratio'])).add(pair['intercept'])
            half_life = estimate_half_life(pair.spread.loc[t: test_end])                

            spread = pair.spread.rolling(window=min(2 * half_life, max_window))
            pair['z_score'] = pair.spread.sub(spread.mean()).div(spread.std())
            pair_out = pair.loc[trading_start: T].assign(s1=y, s2=x, period=p, pair=i).drop([x, y], axis=1)

            hl_out = [test_end, y, x, half_life]
            return pair_out, hl_out

        pair_results = Parallel(n_jobs=-1, verbose=10)(
            delayed(process_pair)(i, y, x)
            for i, (y, x) in enumerate(zip(period_candidates.y, period_candidates.x), 1)
        )

        pairs.extend([pr[0] for pr in pair_results])
        half_lives.extend([pr[1] for pr in pair_results])

    return pairs, half_lives

# %%
pairs, half_lives = get_spread_parallel(candidates, prices)

### Collect Results

#### Half Lives

In [ ]:
hl = pd.DataFrame(half_lives, columns=['test_end', 's1', 's2', 'half_life'])
hl.info()

In [ ]:
hl.half_life.describe()

In [ ]:
hl

In [ ]:
print(hl.half_life.describe())

import plotly.express as px
fig = px.histogram(hl.half_life)
fig.show()

#### Pair Data

In [ ]:
pair_data = pd.concat(pairs)
pair_data.info(show_counts=True)

In [ ]:
pair_data

Plots the price series of a cointegrated pair (s1 and s2) for a given period and pair ID.
    
This function loads the necessary price data and ticker names from 'backtest.h5'.
It filters the pair_data for the specified period and pair_id to identify s1 and s2,
then plots their close prices over the trading dates in that period.

Parameters:
- period (int): The period number (e.g., 1, 2, ..., corresponding to quarterly test periods).
- pair_id (int): The pair identifier within the period (e.g., 1, 2, ... for each candidate pair).
- normalize (bool, optional): If True, normalize prices to start at 100 for easier comparison. Default is False.
- figsize (tuple, optional): Figure size for the plot. Default is (12, 6).

Returns:
- None: Displays the plot.

Explanation:
In statistical arbitrage with cointegrated pairs, visualizing the raw price series of the two assets (e.g., an ETF and a stock)
helps understand their co-movement. Cointegrated pairs tend to move together in the long run, even if they diverge temporarily.
- The left y-axis shows the price of the first asset (s1).
- The right y-axis shows the price of the second asset (s2) for better scaling.
- If normalize=True, prices are rebased to 100 at the start of the period to highlight relative movements, which is useful for spotting divergences.

In [ ]:
def plot_pair_prices(period, pair_id, normalize=False, figsize=(12, 8)):
    # Filter pair_data for the given period and pair_id
    data = pair_data.query('period == @period & pair == @pair_id')
    
    if data.empty:
        print(f"No data found for period {period} and pair {pair_id}.")
        return
    
    # Extract s1 and s2 (the pair tickers)
    s1 = data['s1'].iloc[0]  # Dependent variable (y)
    s2 = data['s2'].iloc[0]  # Independent variable (x)
    
    # Get the date range for this period/pair (trading dates)
    dates = data.index
    
    # Extract prices for s1 and s2 over the date range
    pair_prices = prices.loc[dates, [s1, s2]].dropna()  # Drop any NaNs if present
    
    if pair_prices.empty:
        print(f"No price data available for {s1} and {s2} in the given period.")
        return
    
    # Estimate the price of s1 using hedge_ratio and intercept
    # Since spread = s1 + hedge_ratio * s2 + intercept ≈ 0,
    # estimated_s1 ≈ -hedge_ratio * s2 - intercept
    estimated = -data['hedge_ratio'] * pair_prices[s2] - data['intercept']
    
    # Add estimated prices and spread to the DataFrame
    pair_prices['estimated'] = estimated
    pair_prices['spread'] = data['spread']
    
    # Optionally normalize prices to start at 100
    if normalize:
        pair_prices[[s1, s2, 'estimated']] = (pair_prices[[s1, s2, 'estimated']] / pair_prices[[s1, s2, 'estimated']].iloc[0]) * 100
    
    # Rename columns to use full names for the legend
    pair_prices = pair_prices.rename(columns={
        s1: tickers.get(s1, s1),
        s2: tickers.get(s2, s2),
        'estimated': f"Estimated {tickers.get(s1, s1)}"
    })
    
    # Create the figure with two subplots (stacked vertically)
    fig, (ax1, ax2) = plt.subplots(nrows=2, figsize=figsize, sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    
    # Plot s1, estimated s1, and s2 on the first subplot
    pair_prices[[pair_prices.columns[0], pair_prices.columns[2]]].plot(
        ax=ax1,
        style=['-', '--'],  # Solid for s1, dashed for estimated
        legend=True
    )
    pair_prices[pair_prices.columns[1]].plot(
        ax=ax1,
        secondary_y=True,  # s2 on secondary y-axis
        style='-',  # Solid for s2
        legend=True
    )
    
    # Customize the first subplot
    ax1.set_title(f"Price Series for Pair {pair_id} in Period {period}: {pair_prices.columns[0]} vs {pair_prices.columns[1]} (with Estimated {pair_prices.columns[0]})")
    ax1.set_ylabel("Price" if not normalize else "Normalized Price (starting at 100)")
    ax1.grid(True)
    
    # Plot the spread on the second subplot
    pair_prices['spread'].plot(ax=ax2, color='purple', legend=True)
    ax2.axhline(0, color='black', linestyle='--', linewidth=1)  # Add horizontal line at zero
    ax2.set_title(f"Spread for Pair {pair_id} in Period {period}")
    ax2.set_xlabel("Date")
    ax2.set_ylabel("Spread")
    ax2.grid(True)
    
    # Apply Seaborn despine for cleaner look
    sns.despine()
    plt.tight_layout()
    plt.show()

# Example usage: Plot for period 1, pair 1 (adjust based on your data)
plot_pair_prices(period=1, pair_id=2, normalize=True)

In [ ]:
# Get unique (period, pair) combinations
unique_pairs = pair_data[['period', 'pair']].drop_duplicates()

# Loop through each unique period and pair
for index, row in unique_pairs.iterrows():
    period = row['period']
    pair_id = row['pair']
    print(f"Plotting for period {period}, pair {pair_id}")
    plot_pair_prices(period=period, pair_id=pair_id, normalize=False, figsize=(12, 8))

### Identify Long & Short Entry and Exit Dates

In [ ]:
def get_trades(data):
    pair_trades = []
    for i, ((period, s1, s2), pair) in enumerate(data.groupby(['period', 's1', 's2']), 1):
        if i % 100 == 0:
            print(f"Processing pair {i}: {s1}-{s2}, period {period}")

        # Check if pair has valid data
        if pair.empty or 'z_score' not in pair.columns:
            print(f"Skipping pair {s1}-{s2}, period {period}: Empty or missing z_score")
            continue

        # Define 3-month windows
        try:
            first3m = pair.first('3M').index
            last3m = pair.last('3M').index
        except Exception as e:
            print(f"Skipping pair {s1}-{s2}, period {period}: Error defining 3M windows - {e}")
            continue

        # Generate entry signals (z-score > 2 or < -2)
        entry = pair.z_score.abs() > 2
        entry = ((entry.shift() != entry)
                 .mul(np.sign(pair.z_score))
                 .fillna(0)
                 .astype(int)
                 .sub(2))

        # Generate exit signals (when z-score crosses zero)
        exit = (np.sign(pair.z_score.shift().fillna(method='bfill'))
                != np.sign(pair.z_score)).astype(int) - 1

        # Concatenate entry and exit signals
        trades_df = pd.concat([entry[entry != -2], exit[exit == 0]], axis=0)
        trades_df = trades_df.to_frame('side')

        # Ensure the index has a name for proper reset (standardize to 'date')
        trades_df.index.name = 'date'

        # Reset index to make 'date' a column for sorting
        trades_df = trades_df.reset_index()

        # Verify and standardize date column (handle 'Date', 'index', etc.)
        possible_date_cols = ['date', 'Date', 'index']
        date_col = next((col for col in possible_date_cols if col in trades_df.columns), None)
        if date_col is None:
            print(f"Skipping pair {s1}-{s2}, period {period}: No date-related column found")
            print(trades_df.columns)
            continue
        if date_col != 'date':
            trades_df = trades_df.rename(columns={date_col: 'date'})

        # Sort by date and side
        trades_df = trades_df.sort_values(['date', 'side'])

        # Set 'date' back as index
        trades_df = trades_df.set_index('date')

        # Convert to Series
        trades = trades_df['side'].squeeze()

        if not isinstance(trades, pd.Series):
            print(f"Skipping pair {s1}-{s2}, period {period}: trades is not a Series")
            print(trades)
            continue

        # Adjust trade signals
        try:
            trades.loc[trades < 0] += 2
        except Exception as e:
            print(f"Skipping pair {s1}-{s2}, period {period}: Error adjusting trades - {e}")
            print(pair.z_score.describe())
            continue

        # Remove consecutive duplicate signals
        trades = trades[trades.abs().shift() != trades.abs()]

        # Select trades in the first 3 months
        try:
            window = trades.loc[first3m.min():first3m.max()]
        except Exception as e:
            print(f"Skipping pair {s1}-{s2}, period {period}: Error selecting first3m window - {e}")
            continue

        # Get extra trades in the last 3 months for exit signals
        try:
            extra = trades.loc[last3m.min():last3m.max()]
        except Exception as e:
            print(f"Skipping pair {s1}-{s2}, period {period}: Error selecting last3m window - {e}")
            continue

        n = len(trades)

        # If no trades, skip
        if n == 0:
            print(f"Skipping pair {s1}-{s2}, period {period}: No trades generated")
            continue

        # If the first trade is an exit, skip it if there are multiple trades
        if not window.empty and window.iloc[0] == 0:
            if n > 1:
                print(f"Shifting window for pair {s1}-{s2}, period {period}")
                window = window.iloc[1:]

        # If the last trade is not an exit, include the next exit from extra
        if not window.empty and window.iloc[-1] != 0:
            extra_exits = extra[extra == 0].head(1)
            if extra_exits.empty:
                print(f"Skipping pair {s1}-{s2}, period {period}: No exit signal in extra")
                continue
            else:
                window = pd.concat([window, extra_exits])

        # If window is empty after adjustments, skip
        if window.empty:
            print(f"Skipping pair {s1}-{s2}, period {period}: Empty window after adjustments")
            continue

        # Join trades with pair data
        trades = pair[['s1', 's2', 'hedge_ratio', 'period', 'pair']].join(window.to_frame('side'), how='right')
        trades.loc[trades.side == 0, 'hedge_ratio'] = np.nan
        trades.hedge_ratio = trades.hedge_ratio.ffill()

        # Append to pair_trades
        pair_trades.append(trades)
        print(f"Added trades for pair {s1}-{s2}, period {period}: {len(trades)} rows")

    # Check if pair_trades is empty
    if not pair_trades:
        print("Warning: No trades generated for any pairs")
    else:
        print(f"Generated trades for {len(pair_trades)} pairs")

    return pair_trades

In [ ]:
# Call the function
pair_trades = get_trades(pair_data)

In [ ]:
# Concatenate and display results
if pair_trades:
    pair_trade_data = pd.concat(pair_trades)
    pair_trade_data.info()
    print(pair_trade_data.head())
else:
    print("Error: pair_trades is empty, cannot concatenate")


In [ ]:
# Plot cumulative trades
if pair_trades:
    trades = pair_trade_data['side'].copy()
    trades.loc[trades != 0] = 1
    trades.loc[trades == 0] = -1
    trades.sort_index().cumsum().plot(figsize=(14, 4))
    sns.despine()